<a href="https://colab.research.google.com/github/krgyaan/LLM-Finetuning/blob/main/Preference_Aligned_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### What to Learn
- What is a preference alignment or preference training?
- Why we do preference training.
- Dataset Example (Human Preference Dataset)
- Techniques for Preference Alignment
- DPO Training — Mathematical Formula (not complete intutiton)
- Practical Implementation with DPO (Direct Performance Optimization)
- LoRA Adapters

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-3"

In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)



print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Now lets start with prefrence base tuning or preference based alignment


In [ ]:
!pip install -U trl

In [ ]:
!pip install -U bitsandbytes

In [ ]:
from trl import DPOTrainer
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [ ]:
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

get_peft_model() → Create a new LoRA during training

PeftModel.from_pretrained() → Load an already-trained LoRA for inference or further training

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)